In [ ]:
import sys
sys.path.append('/workspace')


In [ ]:
from mpinn.config import PhysicsParams, TrainConfig, DEFAULT_PHYSICS, DEFAULT_TRAIN_CONFIG
from mpinn.runner import run_experiment, run_grid_search
from mpinn.plotting import show_plot, save_plot, show_history


In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import pandas as pd
import os
from functools import partial


In [ ]:
from mpinn.geom import Sphere
from mpinn.pde import sphere_1d
from mpinn.bc import neuman_bc
from mpinn.analytic import sphere_1d_neuman_exact


In [ ]:
jax.devices()


In [ ]:
plt.rcParams.update({
    'font.size': 14, 'axes.titlesize': 14, 'axes.labelsize': 14,
    'xtick.labelsize': 14, 'ytick.labelsize': 14, 'legend.fontsize': 14, 'figure.titlesize': 16,
})


In [ ]:
# Физические параметры для задачи Неймана (сферическая геометрия)
phys = PhysicsParams(x_left=0.1, x_right=1.0, T_left=500.0, h=10.0, _lambda=1.0)
phys


In [ ]:
base_config = TrainConfig(hidden_features=64, num_layers=2, activation_name='GELU',
    opt_name='adam', lr=0.01, max_epochs=3000, patience=200, num_points=100,
    weights=(1.0, 1.0, 1.0), save_img=False, show_plot=False)
base_config


In [ ]:
# Граничные условия Неймана
bc_fns = [
    partial(neuman_bc, x=jnp.array([phys.x_left]), T=phys.T_left, h=phys.h, _lambda=phys._lambda),
    partial(neuman_bc, x=jnp.array([phys.x_right]), T=None, h=phys.h, _lambda=phys._lambda)
]


In [ ]:
config = TrainConfig(hidden_features=64, num_layers=2, activation_name='GELU',
    opt_name='adam', lr=0.01, max_epochs=3000, patience=200, num_points=100,
    weights=(1.0, 1.0, 1.0), save_img=True, show_plot=True, image_path='images/pinn/1D_sphere_neuman.png')
metrics, history = run_experiment(config=config, phys=phys, pde_fn=sphere_1d,
    exact_fn=sphere_1d_neuman_exact, bc_fns_override=bc_fns)
metrics


In [ ]:
show_history(history, save_path='images/pinn/1D_sphere_neuman_history.png', show_plot=True)


In [ ]:
param_grid = {
    'hidden_features': [32, 64, 128], 'num_layers': [1, 2, 3],
    'activation_name': ['tanh', 'Swish', 'GELU', 'ReLU'], 'num_points': [10, 20, 50, 100],
}


In [ ]:
df_results = run_grid_search(param_grid=param_grid, base_config=base_config, phys=phys,
    csv_path='csv_results/1D_sphere_neuman_results.csv', pde_fn=sphere_1d, exact_fn=sphere_1d_neuman_exact)
df_results.head()
